Prepare data for models

In [1]:
import os
import librosa
import numpy as np
from tqdm import tqdm

DATA_PATH = '../Data/speech_commands/'
SAVE_PATH = '../Data/'

TARGET_WORDS = ['on', 'off', 'stop']
UNKNOWN_WORDS = ['right', 'left', 'up', 'down']

X = [] 
y = [] 

def get_mfcc(file_path):
    try:
        # 1. Load audio file
        audio, sr = librosa.load(file_path, sr=16000)
        
        # 2. FEATURE ENGINEERING: Trim silence from beginning and end
        # top_db=25 means any sound 25dB below the peak volume is considered silence and removed
        trimmed_audio, _ = librosa.effects.trim(audio, top_db=25)
        
        # Filter out extremely short noises (e.g., clicks shorter than 0.15 seconds)
        if len(trimmed_audio) < 2400: 
            return None

        # 3. Extract MFCC on the PURE spoken word
        mfccs = librosa.feature.mfcc(y=trimmed_audio, sr=sr, n_mfcc=13, hop_length=512)
        mfccs = mfccs.T # Transpose to shape (frames, 13)

        # 4. FEATURE ENGINEERING: Dynamic Temporal Pooling
        # Dynamically split the frames into 3 equal sections, regardless of length
        parts = np.array_split(mfccs, 3)
        
        # Calculate the mean for each dynamic part
        part1 = np.mean(parts[0], axis=0) # Always the start of the word
        part2 = np.mean(parts[1], axis=0) # Always the middle
        part3 = np.mean(parts[2], axis=0) # Always the end

        # Concatenate them into a single 1D array of 39 features
        features = np.concatenate((part1, part2, part3))
        
        return features
    except Exception as e:
        return None

print("Processing Target Words (with Silence Trimming)...")
for label_idx, word in enumerate(TARGET_WORDS):
    word_dir = os.path.join(DATA_PATH, word)
    if not os.path.exists(word_dir):
        print(f"Warning: Directory for '{word}' not found.")
        continue
        
    files = [f for f in os.listdir(word_dir) if f.endswith('.wav')]
    for file_name in tqdm(files[:1500], desc=f"Loading {word}"): 
        file_path = os.path.join(word_dir, file_name)
        mfcc_data = get_mfcc(file_path)
        if mfcc_data is not None:
            X.append(mfcc_data)
            y.append(label_idx)

print("\nProcessing Unknown Words (with Silence Trimming)...")
unknown_label_idx = len(TARGET_WORDS) 
for word in UNKNOWN_WORDS:
    word_dir = os.path.join(DATA_PATH, word)
    if not os.path.exists(word_dir):
        continue
        
    files = [f for f in os.listdir(word_dir) if f.endswith('.wav')]
    for file_name in tqdm(files[:375], desc=f"Loading {word} (Unknown)"): 
        file_path = os.path.join(word_dir, file_name)
        mfcc_data = get_mfcc(file_path)
        if mfcc_data is not None:
            X.append(mfcc_data)
            y.append(unknown_label_idx)

X = np.array(X)
y = np.array(y)

np.save(os.path.join(SAVE_PATH, 'X_features.npy'), X)
np.save(os.path.join(SAVE_PATH, 'y_labels.npy'), y)

print(f"\nFeature extraction complete! Saved {len(X)} robust samples.")
print(f"NEW Features shape: {X.shape} (Dynamic Temporal Pooling)")

Processing Target Words (with Silence Trimming)...


Loading on:   0%|          | 0/1500 [00:00<?, ?it/s]c:\Users\User\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
Loading stop: 100%|██████████| 1500/1500 [00:16<00:00, 88.89it/s] 



Processing Unknown Words (with Silence Trimming)...


Loading down (Unknown): 100%|██████████| 375/375 [00:03<00:00, 100.27it/s]


Feature extraction complete! Saved 5995 robust samples.
NEW Features shape: (5995, 39) (Dynamic Temporal Pooling)


RandomForest

In [2]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
# import m2cgen as m2c
import os

DATA_PATH = '../Data/'
OUTPUT_PATH = '../edge_mcu'

# 1. Load the pre-processed features and labels
X = np.load(os.path.join(DATA_PATH, 'X_features.npy'))
y = np.load(os.path.join(DATA_PATH, 'y_labels.npy'))

print(f"Loaded Data -> X shape: {X.shape}, y shape: {y.shape}")

# Split the dataset into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Define the Optuna objective function for a TINY Random Forest model
def objective(trial):
    param = {
        # STRICT LIMITS: Keeping trees small for the MCU's RAM
        'n_estimators': trial.suggest_int('n_estimators', 10, 30), 
        'max_depth': trial.suggest_int('max_depth', 2, 10), 
        'random_state': 42,
        'n_jobs': -1
    }
    
    model = RandomForestClassifier(**param)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    return accuracy_score(y_test, preds)

# 3. Run the hyperparameter optimization
print("\nStarting Optuna optimization for a lightweight model...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print("\nBest parameters found:", study.best_params)
print(f"Best cross-validation accuracy: {study.best_value * 100:.2f}%")

# 4. Train the final Random Forest model
print("\nTraining the final Random Forest model...")
final_model = RandomForestClassifier(**study.best_params, random_state=42)
final_model.fit(X_train, y_train)

# Evaluate the final model
final_preds = final_model.predict(X_test)
print("\nClassification Report on Test Data:")

unique_classes = np.unique(y_test)
target_names = ['on', 'off', 'stop', 'unknown'][:len(unique_classes)]

print(classification_report(y_test, final_preds, target_names=target_names))

# 5. Export the trained model parameters as lightweight arrays for MicroPython
print("\nExporting tree parameters (arrays) for memory-efficient Edge Deployment...")

trees_dump = []
for estimator in final_model.estimators_:
    tree = estimator.tree_
    
    # In Random Forest, tree.value stores the class counts at each node
    # We convert this to the actual predicted class index for leaf nodes
    node_classes = [int(np.argmax(val)) for val in tree.value]
    
    tree_data = {
        'left': tree.children_left.tolist(),
        'right': tree.children_right.tolist(),
        'feature': tree.feature.tolist(),
        # Round thresholds to 4 decimals to save storage space
        'threshold': [round(t, 4) for t in tree.threshold.tolist()], 
        'classes': node_classes
    }
    trees_dump.append(tree_data)

# Save to a Python file as a simple list of dictionaries
output_file = os.path.join(OUTPUT_PATH, 'model_data_randomForest.py')
with open(output_file, 'w') as f:
    f.write("# Auto-generated Random Forest parameters for Edge AI\n")
    f.write("TREES = [\n")
    for t in trees_dump:
        f.write(f"    {t},\n")
    f.write("]\n")

print(f"Model parameters successfully exported to: {output_file}")

[I 2026-07-14 12:12:32,605] A new study created in memory with name: no-name-5d8a3d0f-d5ca-44c5-8689-84dbba1446d7
[I 2026-07-14 12:12:32,743] Trial 0 finished with value: 0.755629691409508 and parameters: {'n_estimators': 22, 'max_depth': 7}. Best is trial 0 with value: 0.755629691409508.


Loaded Data -> X shape: (5995, 39), y shape: (5995,)

Starting Optuna optimization for a lightweight model...


[I 2026-07-14 12:12:32,834] Trial 1 finished with value: 0.6713928273561302 and parameters: {'n_estimators': 25, 'max_depth': 2}. Best is trial 0 with value: 0.755629691409508.
[I 2026-07-14 12:12:32,926] Trial 2 finished with value: 0.7030859049207673 and parameters: {'n_estimators': 22, 'max_depth': 4}. Best is trial 0 with value: 0.755629691409508.
[I 2026-07-14 12:12:33,033] Trial 3 finished with value: 0.6505421184320267 and parameters: {'n_estimators': 15, 'max_depth': 2}. Best is trial 0 with value: 0.755629691409508.
[I 2026-07-14 12:12:33,148] Trial 4 finished with value: 0.7481234361968306 and parameters: {'n_estimators': 28, 'max_depth': 6}. Best is trial 0 with value: 0.755629691409508.
[I 2026-07-14 12:12:33,252] Trial 5 finished with value: 0.7589658048373644 and parameters: {'n_estimators': 20, 'max_depth': 8}. Best is trial 5 with value: 0.7589658048373644.
[I 2026-07-14 12:12:33,366] Trial 6 finished with value: 0.7848206839032527 and parameters: {'n_estimators': 24, '


Best parameters found: {'n_estimators': 30, 'max_depth': 9}
Best cross-validation accuracy: 78.82%

Training the final Random Forest model...

Classification Report on Test Data:
              precision    recall  f1-score   support

          on       0.78      0.80      0.79       299
         off       0.80      0.80      0.80       300
        stop       0.90      0.82      0.86       300
     unknown       0.69      0.74      0.72       300

    accuracy                           0.79      1199
   macro avg       0.79      0.79      0.79      1199
weighted avg       0.79      0.79      0.79      1199


Exporting tree parameters (arrays) for memory-efficient Edge Deployment...
Model parameters successfully exported to: ../edge_mcu\model_data_randomForest.py


LogisticRegression

In [3]:
import numpy as np
from sklearn.linear_model import LogisticRegression
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import os

DATA_PATH = '../Data/'
OUTPUT_PATH = '../edge_mcu'

# 1. Load the pre-processed features and labels
X = np.load(os.path.join(DATA_PATH, 'X_features.npy'))
y = np.load(os.path.join(DATA_PATH, 'y_labels.npy'))

print(f"Loaded Data -> X shape: {X.shape}, y shape: {y.shape}")

# Split the dataset into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Define the Optuna objective function for Logistic Regression
def objective(trial):
    param = {
        # 'C' is the inverse of regularization strength. 
        # Smaller values specify stronger limits/constraints on the weights.
        'C': trial.suggest_float('C', 1e-4, 1e2, log=True),
        
        # 'solver' is the algorithm used to optimize the weights
        'solver': trial.suggest_categorical('solver', ['lbfgs', 'newton-cg', 'saga']),
        
        # Max_iter is kept high to allow the mathematical solver to fully converge
        'max_iter': 2000, 
        'random_state': 42,
        'n_jobs': -1
    }
    
    model = LogisticRegression(**param)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    return accuracy_score(y_test, preds)

# 3. Run the hyperparameter optimization
print("\nStarting Optuna optimization for Logistic Regression...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print("\nBest parameters found:", study.best_params)
print(f"Best cross-validation accuracy: {study.best_value * 100:.2f}%")

# 4. Train the final model using the exact optimized parameters
print("\nTraining the final Logistic Regression model...")
final_model = LogisticRegression(**study.best_params, max_iter=2000, random_state=42)
final_model.fit(X_train, y_train)

# Evaluate the final model
final_preds = final_model.predict(X_test)
print("\nClassification Report on Test Data:")

# Dynamically extract classes
unique_classes = np.unique(y_test)
target_names = ['on', 'off', 'stop', 'unknown'][:len(unique_classes)]
print(classification_report(y_test, final_preds, target_names=target_names))

# 5. Export weights and intercepts to a tiny Python file
print("\nExporting model parameters (Weights & Intercepts) for Edge Deployment...")

weights = final_model.coef_.tolist()
intercepts = final_model.intercept_.tolist()

output_file = os.path.join(OUTPUT_PATH, 'model_data_regression.py')
with open(output_file, 'w') as f:
    f.write("# Auto-Generated Lightweight Model for Keyword Spotting\n\n")
    f.write(f"CLASSES = {target_names}\n\n")
    
    # We round the numbers to 4 decimal places to save even more space on the MCU
    f.write("WEIGHTS = [\n")
    for class_weights in weights:
        rounded_weights = [round(w, 4) for w in class_weights]
        f.write(f"    {rounded_weights},\n")
    f.write("]\n\n")
    
    rounded_intercepts = [round(i, 4) for i in intercepts]
    f.write(f"INTERCEPTS = {rounded_intercepts}\n")

print(f"Model successfully exported to: {output_file}")
print("Done! The model_data.py is now highly optimized and incredibly lightweight.")

[I 2026-07-14 12:12:40,306] A new study created in memory with name: no-name-fb54a428-bb58-40a8-8f15-f1cb81e46b32


Loaded Data -> X shape: (5995, 39), y shape: (5995,)

Starting Optuna optimization for Logistic Regression...


c:\Users\User\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
[I 2026-07-14 12:12:41,344] Trial 0 finished with value: 0.7898248540450375 and parameters: {'C': 0.048597520287432486, 'solver': 'newton-cg'}. Best is trial 0 with value: 0.7898248540450375.
c:\Users\User\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
[I 2026-07-14 12:12:43,135] Trial 1 finished with value: 0.7948290241868223 and parameters: {'C': 36.85178585180183, 'solver': 'saga'}. Best is trial 1 with value: 0.7948290241868223.
c:\Users\User\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' 


Best parameters found: {'C': 36.85178585180183, 'solver': 'saga'}
Best cross-validation accuracy: 79.48%

Training the final Logistic Regression model...

Classification Report on Test Data:
              precision    recall  f1-score   support

          on       0.79      0.82      0.80       299
         off       0.82      0.79      0.80       300
        stop       0.88      0.82      0.85       300
     unknown       0.70      0.75      0.73       300

    accuracy                           0.79      1199
   macro avg       0.80      0.79      0.80      1199
weighted avg       0.80      0.79      0.80      1199


Exporting model parameters (Weights & Intercepts) for Edge Deployment...
Model successfully exported to: ../edge_mcu\model_data_regression.py
Done! The model_data.py is now highly optimized and incredibly lightweight.


In [4]:
import sys
import os
import random

# Ensure the output directory is in the system path
sys.path.append(os.path.abspath('../edge_mcu'))
import model_data_regression as model_data

def predict_audio_class(mfcc_features):
    """
    Computes the dot product of features and weights for each class,
    adds the intercept, and returns the class with the highest score.
    Optimized for memory-efficient edge inference.
    """
    num_classes = len(model_data.INTERCEPTS)
    num_features = len(mfcc_features)
    
    class_scores = [0.0] * num_classes
    
    # Calculate score for each class
    for c in range(num_classes):
        score = model_data.INTERCEPTS[c]
        # Dot product: multiply each feature by its corresponding weight
        for j in range(num_features):
            score += mfcc_features[j] * model_data.WEIGHTS[c][j]
        class_scores[c] = score
        
    # Find the index of the highest score (Argmax)
    best_class_idx = 0
    max_score = class_scores[0]
    for c in range(1, num_classes):
        if class_scores[c] > max_score:
            max_score = class_scores[c]
            best_class_idx = c
            
    return best_class_idx

# --- Random Evaluation on Test Data ---
print("Evaluating Logistic Regression on edge-like inference:")
for _ in range(5):
    idx = random.randint(0, len(y_test) - 1)
    features = X_test[idx]
    
    result_idx = predict_audio_class(features)
    predicted_word = model_data.CLASSES[result_idx]
    actual_word = model_data.CLASSES[y_test[idx]]
    
    print(f"Predicted: {predicted_word: <10} | Actual: {actual_word}")

Evaluating Logistic Regression on edge-like inference:
Predicted: unknown    | Actual: off
Predicted: on         | Actual: on
Predicted: stop       | Actual: stop
Predicted: on         | Actual: on
Predicted: unknown    | Actual: unknown


XGBoost

In [5]:
import numpy as np
import xgboost as xgb
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import os
import json

DATA_PATH = '../Data/'
OUTPUT_PATH = '../edge_mcu'

# 1. Load the pre-processed features and labels
X = np.load(os.path.join(DATA_PATH, 'X_features.npy'))
y = np.load(os.path.join(DATA_PATH, 'y_labels.npy'))

print(f"Loaded Data -> X shape: {X.shape}, y shape: {y.shape}")

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Define Optuna objective for XGBoost
def objective(trial):
    param = {
        'objective': 'multi:softprob',
        'max_depth': trial.suggest_int('max_depth', 2, 3), 
        'n_estimators': trial.suggest_int('n_estimators', 2, 5), 
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'random_state': 42,
        'n_jobs': -1
    }
    
    model = xgb.XGBClassifier(**param)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    return accuracy_score(y_test, preds)

# 3. Optimize Hyperparameters
print("\nStarting Optuna optimization for XGBoost...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print("\nBest parameters found:", study.best_params)
print(f"Best cross-validation accuracy: {study.best_value * 100:.2f}%")

# 4. Train the Final XGBoost Model
print("\nTraining the final XGBoost model...")
best_params = study.best_params
best_params['objective'] = 'multi:softprob'
best_params['random_state'] = 42

final_model = xgb.XGBClassifier(**best_params)
final_model.fit(X_train, y_train)

# Evaluate the final model
final_preds = final_model.predict(X_test)
print("\nClassification Report on Test Data:")
unique_classes = np.unique(y_test)
target_names = ['on', 'off', 'stop', 'unknown'][:len(unique_classes)]
print(classification_report(y_test, final_preds, target_names=target_names))

# 5. CUSTOM EXPORT: Bypass m2cgen and extract trees directly
print("\nExporting XGBoost trees manually for ultra-lightweight Edge Deployment...")

booster = final_model.get_booster()
# Dump all trees to JSON format
trees_json = booster.get_dump(dump_format='json')

parsed_trees = []
for tree_str in trees_json:
    tree_dict = json.loads(tree_str)
    
    # Recursive function to find the maximum node ID to size our arrays
    def get_max_nodeid(node):
        max_id = node.get('nodeid', 0)
        if 'children' in node:
            for child in node['children']:
                max_id = max(max_id, get_max_nodeid(child))
        return max_id
        
    size = get_max_nodeid(tree_dict) + 1
    
    # Initialize flattened arrays for MicroPython
    features = [-1] * size
    thresholds = [0.0] * size
    lefts = [-1] * size
    rights = [-1] * size
    values = [0.0] * size
    
    # Recursive function to parse JSON into flat arrays
    def parse_node(node):
        nid = node['nodeid']
        if 'leaf' in node:
            values[nid] = round(node['leaf'], 4)
        else:
            # XGBoost features are formatted as 'f0', 'f1', etc. We extract the integer.
            feat_str = node['split']
            features[nid] = int(feat_str.replace('f', ''))
            thresholds[nid] = round(node['split_condition'], 4)
            lefts[nid] = node['yes']
            rights[nid] = node['no']
            for child in node['children']:
                parse_node(child)
                
    parse_node(tree_dict)
    
    parsed_trees.append({
        'features': features,
        'thresholds': thresholds,
        'left': lefts,
        'right': rights,
        'values': values
    })

# Save the parsed structure to a Python file
output_file = os.path.join(OUTPUT_PATH, 'model_data_xgb.py')
with open(output_file, 'w') as f:
    f.write("# Auto-Generated Custom XGBoost Export for MicroPython\n\n")
    f.write(f"CLASSES = {target_names}\n")
    f.write(f"NUM_CLASS = {len(target_names)}\n\n")
    f.write("TREES = [\n")
    for t in parsed_trees:
        f.write(f"    {t},\n")
    f.write("]\n")

print(f"XGBoost Model successfully exported to: {output_file}")

[I 2026-07-14 12:13:38,690] A new study created in memory with name: no-name-4f245c2d-d480-4c27-b4e4-b149cebf58f4
[I 2026-07-14 12:13:38,773] Trial 0 finished with value: 0.640533778148457 and parameters: {'max_depth': 3, 'n_estimators': 2, 'learning_rate': 0.07090028466413667}. Best is trial 0 with value: 0.640533778148457.
[I 2026-07-14 12:13:38,829] Trial 1 finished with value: 0.640533778148457 and parameters: {'max_depth': 2, 'n_estimators': 4, 'learning_rate': 0.25612441237093236}. Best is trial 0 with value: 0.640533778148457.
[I 2026-07-14 12:13:38,869] Trial 2 finished with value: 0.6105087572977481 and parameters: {'max_depth': 2, 'n_estimators': 3, 'learning_rate': 0.15191155032736256}. Best is trial 0 with value: 0.640533778148457.


Loaded Data -> X shape: (5995, 39), y shape: (5995,)

Starting Optuna optimization for XGBoost...


[I 2026-07-14 12:13:38,906] Trial 3 finished with value: 0.6121768140116765 and parameters: {'max_depth': 2, 'n_estimators': 3, 'learning_rate': 0.17701354038628536}. Best is trial 0 with value: 0.640533778148457.
[I 2026-07-14 12:13:38,952] Trial 4 finished with value: 0.6255212677231026 and parameters: {'max_depth': 2, 'n_estimators': 5, 'learning_rate': 0.09348111352106951}. Best is trial 0 with value: 0.640533778148457.
[I 2026-07-14 12:13:39,001] Trial 5 finished with value: 0.6605504587155964 and parameters: {'max_depth': 3, 'n_estimators': 4, 'learning_rate': 0.0980477629119088}. Best is trial 5 with value: 0.6605504587155964.
[I 2026-07-14 12:13:39,028] Trial 6 finished with value: 0.6447039199332777 and parameters: {'max_depth': 2, 'n_estimators': 3, 'learning_rate': 0.2742374970313965}. Best is trial 5 with value: 0.6605504587155964.
[I 2026-07-14 12:13:39,063] Trial 7 finished with value: 0.6680567139282736 and parameters: {'max_depth': 3, 'n_estimators': 3, 'learning_rate':


Best parameters found: {'max_depth': 3, 'n_estimators': 5, 'learning_rate': 0.2955571860575909}
Best cross-validation accuracy: 70.98%

Training the final XGBoost model...

Classification Report on Test Data:
              precision    recall  f1-score   support

          on       0.68      0.70      0.69       299
         off       0.75      0.77      0.76       300
        stop       0.86      0.70      0.78       300
     unknown       0.58      0.66      0.62       300

    accuracy                           0.71      1199
   macro avg       0.72      0.71      0.71      1199
weighted avg       0.72      0.71      0.71      1199


Exporting XGBoost trees manually for ultra-lightweight Edge Deployment...
XGBoost Model successfully exported to: ../edge_mcu\model_data_xgb.py


In [6]:
import sys
import os

sys.path.append(os.path.abspath('../edge_mcu'))

import model_data_xgb as model_data
def predict_audio_class(mfcc_features):
    """
    Traverses the exported XGBoost trees using the extracted MFCC features.
    Returns the index of the predicted class based on the highest raw score.
    """
    num_class = model_data.NUM_CLASS
    
    # Array to accumulate scores (raw logits) for each class
    class_scores = [0.0] * num_class
    
    # In multi-class XGBoost, trees are assigned to classes in a round-robin fashion.
    # Tree 0 -> Class 0, Tree 1 -> Class 1 ... Tree 4 -> Class 0, etc.
    for i, tree in enumerate(model_data.TREES):
        node = 0 # Start at the root of the tree
        
        # Traverse until a leaf is reached (marked by feature index -1)
        while tree['features'][node] != -1:
            feat_idx = tree['features'][node]
            
            # XGBoost splitting rule: strictly less than (<)
            if mfcc_features[feat_idx] < tree['thresholds'][node]:
                node = tree['left'][node]
            else:
                node = tree['right'][node]
                
        # Leaf reached: get the value and add it to the corresponding class score
        leaf_value = tree['values'][node]
        class_idx = i % num_class
        class_scores[class_idx] += leaf_value
        
    # Find the index of the highest score (Argmax)
    best_class = 0
    max_score = class_scores[0]
    for i in range(1, num_class):
        if class_scores[i] > max_score:
            max_score = class_scores[i]
            best_class = i
            
    return best_class

# --- Example Usage on Board ---
# ['on', 'off', 'stop', 'uknown']
i = 16
features = [-2.5686844e+02,  1.3959177e+02,  8.0665617e+00, -4.1448826e+01, -2.9944046e+01,  3.2988396e-01,  6.2178426e+00,  9.0643892e+00, -6.5838027e+00,  5.9275532e+00, -2.8192373e+01,  1.9626240e+01, -1.0130860e+00, -2.5470102e+02,  1.6171916e+02, -9.4717093e+00, -1.6660915e+01,  9.5316715e+00,  1.5823692e+01, -4.0831656e+00, -1.6109245e+01, -1.6403475e+01, -2.7065601e+00, -7.6615257e+00, 1.2409652e+01, -2.4190521e+01, -4.9443311e+02,  8.9877983e+01, 1.8117041e+01,  3.1036968e+01,  5.0818684e+01,  3.1483164e+01, 4.5603323e+00, -1.2117879e+01, -1.4944232e+01,  2.7241716e+00, 7.9998846e+00, -1.2753278e+01, -1.6924435e+01] # 13 features from mic
result_idx = predict_audio_class(features)
print("Predicted Word:", model_data.CLASSES[result_idx], "and actuall is:", y_test[i])


Predicted Word: on and actuall is: 1


*now with Root reduction and feature importance*

In [7]:
import os
import librosa
import numpy as np
from tqdm import tqdm

DATA_PATH = '../Data/speech_commands/'
SAVE_PATH = '../Data/'

TARGET_WORDS = ['on', 'off', 'stop']
UNKNOWN_WORDS = ['right', 'left', 'up', 'down']

X = [] 
y = [] 

def get_mfcc(file_path):
    try:
        audio, sr = librosa.load(file_path, sr=16000)
        
        # Trim silence
        trimmed_audio, _ = librosa.effects.trim(audio, top_db=25)
        if len(trimmed_audio) < 2400: 
            return None

        # ROOT REDUCTION: Extract only 8 MFCCs instead of 13
        mfccs = librosa.feature.mfcc(y=trimmed_audio, sr=sr, n_mfcc=8, hop_length=512)
        mfccs = mfccs.T 

        # Dynamic Temporal Pooling (3 parts)
        parts = np.array_split(mfccs, 3)
        
        part1 = np.mean(parts[0], axis=0) 
        part2 = np.mean(parts[1], axis=0) 
        part3 = np.mean(parts[2], axis=0) 

        # Concatenate: 8 + 8 + 8 = 24 features
        features = np.concatenate((part1, part2, part3))
        
        return features
    except Exception as e:
        return None

print("Processing Target Words (with 8-MFCC Root Reduction)...")
for label_idx, word in enumerate(TARGET_WORDS):
    word_dir = os.path.join(DATA_PATH, word)
    if not os.path.exists(word_dir):
        continue
        
    files = [f for f in os.listdir(word_dir) if f.endswith('.wav')]
    for file_name in tqdm(files[:1500], desc=f"Loading {word}"): 
        file_path = os.path.join(word_dir, file_name)
        mfcc_data = get_mfcc(file_path)
        if mfcc_data is not None:
            X.append(mfcc_data)
            y.append(label_idx)

print("\nProcessing Unknown Words...")
unknown_label_idx = len(TARGET_WORDS) 
for word in UNKNOWN_WORDS:
    word_dir = os.path.join(DATA_PATH, word)
    if not os.path.exists(word_dir):
        continue
        
    files = [f for f in os.listdir(word_dir) if f.endswith('.wav')]
    for file_name in tqdm(files[:375], desc=f"Loading {word} (Unknown)"): 
        file_path = os.path.join(word_dir, file_name)
        mfcc_data = get_mfcc(file_path)
        if mfcc_data is not None:
            X.append(mfcc_data)
            y.append(unknown_label_idx)

X = np.array(X)
y = np.array(y)

np.save(os.path.join(SAVE_PATH, 'X_features.npy'), X)
np.save(os.path.join(SAVE_PATH, 'y_labels.npy'), y)

print(f"\nFeature extraction complete! Saved {len(X)} samples.")
print(f"NEW Features shape: {X.shape} (24 features per sample)")

Processing Target Words (with 8-MFCC Root Reduction)...


Loading stop: 100%|██████████| 1500/1500 [00:05<00:00, 260.06it/s]



Processing Unknown Words...


Loading down (Unknown): 100%|██████████| 375/375 [00:01<00:00, 236.38it/s]


Feature extraction complete! Saved 5995 samples.
NEW Features shape: (5995, 24) (24 features per sample)


In [8]:
import numpy as np
import xgboost as xgb
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import os
import json

DATA_PATH = '../Data/'
OUTPUT_PATH = '../edge_mcu'

# 1. Load Data
X = np.load(os.path.join(DATA_PATH, 'X_features.npy'))
y = np.load(os.path.join(DATA_PATH, 'y_labels.npy'))
print(f"Original Data -> X shape: {X.shape}, y shape: {y.shape}")

X_train_full, X_test_full, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. FEATURE SELECTION (Method 1)
print("\n--- Running Feature Selection ---")
# Train a temporary, unconstrained model to find feature importance
temp_model = xgb.XGBClassifier(objective='multi:softprob', random_state=42)
temp_model.fit(X_train_full, y_train)

# Get importance scores and select the top 12 features (out of 24)
importances = temp_model.feature_importances_
# argsort sorts ascending, so we take the last 12 elements for the highest scores
top_12_indices = np.argsort(importances)[-12:] 
top_12_indices = np.sort(top_12_indices) # Sort indices to keep the original order

print(f"Selected Top 12 Feature Indices: {top_12_indices.tolist()}")

# Filter the datasets to ONLY include these 12 features
X_train = X_train_full[:, top_12_indices]
X_test = X_test_full[:, top_12_indices]
print(f"Reduced Data -> X_train shape: {X_train.shape}")

# 3. OPTUNA OPTIMIZATION (on the reduced 12-feature dataset)
print("\n--- Starting Optuna optimization on reduced features ---")
def objective(trial):
    param = {
        'objective': 'multi:softprob',
        # Keep trees extremely shallow for Edge RAM
        'max_depth': trial.suggest_int('max_depth', 2, 3), 
        'n_estimators': trial.suggest_int('n_estimators', 2, 5), 
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'random_state': 42,
        'n_jobs': -1
    }
    
    model = xgb.XGBClassifier(**param)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return accuracy_score(y_test, preds)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)
print(f"\nBest Optuna Accuracy: {study.best_value * 100:.2f}%")

# 4. TRAIN FINAL MODEL
print("\n--- Training Final Lightweight XGBoost ---")
best_params = study.best_params
best_params['objective'] = 'multi:softprob'
best_params['random_state'] = 42

final_model = xgb.XGBClassifier(**best_params)
final_model.fit(X_train, y_train)

preds = final_model.predict(X_test)
unique_classes = np.unique(y_test)
target_names = ['on', 'off', 'stop', 'unknown'][:len(unique_classes)]
print("\nFinal Classification Report:")
print(classification_report(y_test, preds, target_names=target_names))

# 5. CUSTOM EXPORT TO PYTHON ARRAYS
print("\n--- Exporting arrays to model_data.py ---")
booster = final_model.get_booster()
trees_json = booster.get_dump(dump_format='json')

parsed_trees = []
for tree_str in trees_json:
    tree_dict = json.loads(tree_str)
    
    def get_max_nodeid(node):
        max_id = node.get('nodeid', 0)
        if 'children' in node:
            for child in node['children']:
                max_id = max(max_id, get_max_nodeid(child))
        return max_id
        
    size = get_max_nodeid(tree_dict) + 1
    features = [-1] * size
    thresholds = [0.0] * size
    lefts = [-1] * size
    rights = [-1] * size
    values = [0.0] * size
    
    def parse_node(node):
        nid = node['nodeid']
        if 'leaf' in node:
            values[nid] = round(node['leaf'], 4)
        else:
            feat_str = node['split']
            features[nid] = int(feat_str.replace('f', ''))
            thresholds[nid] = round(node['split_condition'], 4)
            lefts[nid] = node['yes']
            rights[nid] = node['no']
            for child in node['children']:
                parse_node(child)
                
    parse_node(tree_dict)
    
    parsed_trees.append({
        'features': features,
        'thresholds': thresholds,
        'left': lefts,
        'right': rights,
        'values': values
    })

output_file = os.path.join(OUTPUT_PATH, 'model_data_xgb_new.py')
with open(output_file, 'w') as f:
    f.write("# Auto-Generated Optimized Custom XGBoost Export\n\n")
    f.write(f"CLASSES = {target_names}\n")
    f.write(f"NUM_CLASS = {len(target_names)}\n")
    # We MUST save the selected feature indices so the MCU knows which features to pass to the model!
    f.write(f"SELECTED_FEATURES = {top_12_indices.tolist()}\n\n") 
    f.write("TREES = [\n")
    for t in parsed_trees:
        f.write(f"    {t},\n")
    f.write("]\n")

print(f"Extremely lightweight XGBoost successfully exported to: {output_file}")

Original Data -> X shape: (5995, 24), y shape: (5995,)

--- Running Feature Selection ---


[I 2026-07-14 12:14:11,856] A new study created in memory with name: no-name-c8263b1d-6804-4124-9d28-da90572e4b91
[I 2026-07-14 12:14:11,919] Trial 0 finished with value: 0.6005004170141784 and parameters: {'max_depth': 2, 'n_estimators': 4, 'learning_rate': 0.022314963804635872}. Best is trial 0 with value: 0.6005004170141784.
[I 2026-07-14 12:14:12,003] Trial 1 finished with value: 0.6563803169307757 and parameters: {'max_depth': 2, 'n_estimators': 4, 'learning_rate': 0.19789215524048898}. Best is trial 1 with value: 0.6563803169307757.


Selected Top 12 Feature Indices: [1, 3, 4, 6, 9, 11, 14, 15, 17, 19, 20, 22]
Reduced Data -> X_train shape: (4796, 12)

--- Starting Optuna optimization on reduced features ---


[I 2026-07-14 12:14:12,121] Trial 2 finished with value: 0.664720600500417 and parameters: {'max_depth': 3, 'n_estimators': 2, 'learning_rate': 0.2708752481293542}. Best is trial 2 with value: 0.664720600500417.
[I 2026-07-14 12:14:12,172] Trial 3 finished with value: 0.5954962468723937 and parameters: {'max_depth': 2, 'n_estimators': 2, 'learning_rate': 0.11064799036862649}. Best is trial 2 with value: 0.664720600500417.
[I 2026-07-14 12:14:12,220] Trial 4 finished with value: 0.6522101751459549 and parameters: {'max_depth': 3, 'n_estimators': 3, 'learning_rate': 0.19062627300456034}. Best is trial 2 with value: 0.664720600500417.
[I 2026-07-14 12:14:12,252] Trial 5 finished with value: 0.6146788990825688 and parameters: {'max_depth': 2, 'n_estimators': 4, 'learning_rate': 0.10100287715316329}. Best is trial 2 with value: 0.664720600500417.
[I 2026-07-14 12:14:12,279] Trial 6 finished with value: 0.6480400333611342 and parameters: {'max_depth': 2, 'n_estimators': 4, 'learning_rate': 0


Best Optuna Accuracy: 69.22%

--- Training Final Lightweight XGBoost ---

Final Classification Report:
              precision    recall  f1-score   support

          on       0.66      0.67      0.66       299
         off       0.73      0.76      0.75       300
        stop       0.86      0.70      0.77       300
     unknown       0.56      0.64      0.60       300

    accuracy                           0.69      1199
   macro avg       0.70      0.69      0.70      1199
weighted avg       0.70      0.69      0.70      1199


--- Exporting arrays to model_data.py ---
Extremely lightweight XGBoost successfully exported to: ../edge_mcu\model_data_xgb_new.py


*Gaussian Naive Bayes*

In [16]:
import numpy as np
from sklearn.naive_bayes import GaussianNB
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import os

DATA_PATH = '../Data/'
OUTPUT_PATH = '../edge_mcu'

# 1. Load the FULL 39-feature dataset
X = np.load(os.path.join(DATA_PATH, 'X_features.npy'))
y = np.load(os.path.join(DATA_PATH, 'y_labels.npy'))

print(f"Loaded Data -> X shape: {X.shape}, y shape: {y.shape}")

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. OPTUNA OPTIMIZATION FOR NAIVE BAYES
print("\n--- Starting Optuna optimization for var_smoothing ---")

def objective(trial):
    # Suggest a value for var_smoothing on a log scale (from very small to relatively large)
    var_smoothing = trial.suggest_float('var_smoothing', 1e-11, 1e-1, log=True)
    
    gnb = GaussianNB(var_smoothing=var_smoothing)
    gnb.fit(X_train, y_train)
    preds = gnb.predict(X_test)
    
    return accuracy_score(y_test, preds)

study = optuna.create_study(direction='maximize')
# GNB is incredibly fast, so we can easily run 100 trials in seconds!
study.optimize(objective, n_trials=100) 

print(f"\nBest Optuna var_smoothing: {study.best_params['var_smoothing']}")
print(f"Best Accuracy during CV: {study.best_value * 100:.2f}%")

# 3. TRAIN FINAL MODEL WITH BEST PARAMETERS
print("\n--- Training Final Optimized Gaussian Naive Bayes ---")
final_gnb = GaussianNB(**study.best_params)
final_gnb.fit(X_train, y_train)

preds = final_gnb.predict(X_test)
unique_classes = np.unique(y_test)
target_names = ['on', 'off', 'stop', 'unknown'][:len(unique_classes)]

print("\nFinal Classification Report:")
print(classification_report(y_test, preds, target_names=target_names))

# 4. EXPORT TO PURE PYTHON ARRAYS (Same size, higher accuracy!)
print("\n--- Exporting Statistical Parameters to model_data_gnb.py ---")

class_priors = final_gnb.class_prior_
if class_priors is None:
    class_counts = np.bincount(y_train)
    class_priors = class_counts / len(y_train)

priors_list = class_priors.tolist()
means_list = final_gnb.theta_.tolist()
vars_list = final_gnb.var_.tolist()

output_file = os.path.join(OUTPUT_PATH, 'model_data_gnb.py')
with open(output_file, 'w') as f:
    f.write("# Auto-Generated Optuna-Optimized Gaussian Naive Bayes Export\n\n")
    f.write(f"CLASSES = {target_names}\n")
    
    pure_priors = [round(float(p), 6) for p in priors_list]
    f.write(f"PRIORS = {pure_priors}\n\n")
    
    f.write("MEANS = [\n")
    for class_means in means_list:
        pure_means = [round(float(m), 6) for m in class_means]
        f.write(f"    {pure_means},\n")
    f.write("]\n\n")
    
    # Notice we don't need to manually add epsilon here anymore, 
    # the optimal var_smoothing already handled the math stabilization!
    f.write("VARIANCES = [\n")
    for class_vars in vars_list:
        pure_vars = [round(float(v), 8) for v in class_vars]
        f.write(f"    {pure_vars},\n")
    f.write("]\n")

print(f"Optimized Model successfully exported to: {output_file}")

[I 2026-07-14 12:21:34,378] A new study created in memory with name: no-name-f2f4237f-e529-4cca-b747-127da639f9e2
[I 2026-07-14 12:21:34,384] Trial 0 finished with value: 0.6814011676396997 and parameters: {'var_smoothing': 0.045536599134380726}. Best is trial 0 with value: 0.6814011676396997.
[I 2026-07-14 12:21:34,389] Trial 1 finished with value: 0.7039199332777314 and parameters: {'var_smoothing': 2.695835682854662e-08}. Best is trial 1 with value: 0.7039199332777314.


[I 2026-07-14 12:21:34,394] Trial 2 finished with value: 0.7039199332777314 and parameters: {'var_smoothing': 9.983923903887797e-05}. Best is trial 1 with value: 0.7039199332777314.
[I 2026-07-14 12:21:34,398] Trial 3 finished with value: 0.7039199332777314 and parameters: {'var_smoothing': 2.7321989623812897e-08}. Best is trial 1 with value: 0.7039199332777314.
[I 2026-07-14 12:21:34,402] Trial 4 finished with value: 0.7039199332777314 and parameters: {'var_smoothing': 1.0293788655515576e-10}. Best is trial 1 with value: 0.7039199332777314.
[I 2026-07-14 12:21:34,406] Trial 5 finished with value: 0.7039199332777314 and parameters: {'var_smoothing': 0.00022384695727108042}. Best is trial 1 with value: 0.7039199332777314.
[I 2026-07-14 12:21:34,409] Trial 6 finished with value: 0.7039199332777314 and parameters: {'var_smoothing': 3.5691800550022765e-05}. Best is trial 1 with value: 0.7039199332777314.
[I 2026-07-14 12:21:34,414] Trial 7 finished with value: 0.7039199332777314 and parame

Loaded Data -> X shape: (5995, 24), y shape: (5995,)

--- Starting Optuna optimization for var_smoothing ---


[I 2026-07-14 12:21:34,581] Trial 35 finished with value: 0.7039199332777314 and parameters: {'var_smoothing': 6.842772694090819e-05}. Best is trial 9 with value: 0.7055879899916597.
[I 2026-07-14 12:21:34,587] Trial 36 finished with value: 0.6688907422852377 and parameters: {'var_smoothing': 0.08645867827494273}. Best is trial 9 with value: 0.7055879899916597.
[I 2026-07-14 12:21:34,592] Trial 37 finished with value: 0.7047539616346956 and parameters: {'var_smoothing': 0.003570343076530397}. Best is trial 9 with value: 0.7055879899916597.
[I 2026-07-14 12:21:34,597] Trial 38 finished with value: 0.7039199332777314 and parameters: {'var_smoothing': 0.00022379533391934562}. Best is trial 9 with value: 0.7055879899916597.
[I 2026-07-14 12:21:34,602] Trial 39 finished with value: 0.7022518765638032 and parameters: {'var_smoothing': 0.012989036157207404}. Best is trial 9 with value: 0.7055879899916597.
[I 2026-07-14 12:21:34,608] Trial 40 finished with value: 0.7039199332777314 and paramet


Best Optuna var_smoothing: 0.0009718160004665792
Best Accuracy during CV: 70.56%

--- Training Final Optimized Gaussian Naive Bayes ---

Final Classification Report:
              precision    recall  f1-score   support

          on       0.64      0.74      0.68       299
         off       0.74      0.67      0.70       300
        stop       0.83      0.76      0.79       300
     unknown       0.64      0.66      0.65       300

    accuracy                           0.71      1199
   macro avg       0.71      0.71      0.71      1199
weighted avg       0.71      0.71      0.71      1199


--- Exporting Statistical Parameters to model_data_gnb.py ---
Optimized Model successfully exported to: ../edge_mcu\model_data_gnb.py


In [17]:
import math
import sys
import os

sys.path.append(os.path.abspath('../edge_mcu'))

import model_data_gnb as model_data
def predict_audio_class(mfcc_features):
    """
    Computes the Gaussian Log-Probability for each class and returns
    the index of the class with the highest probability.
    """
    best_class = 0
    # Initialize with negative infinity
    max_log_prob = -float('inf') 
    
    num_classes = len(model_data.CLASSES)
    num_features = len(mfcc_features)
    
    for c in range(num_classes):
        # Start with the log of the prior probability for this class
        log_prob = math.log(model_data.PRIORS[c])
        
        # Add the log probability of each feature given the class
        for i in range(num_features):
            x = mfcc_features[i]
            mean = model_data.MEANS[c][i]
            var = model_data.VARIANCES[c][i]
            
            # Gaussian Log-PDF Formula: 
            # -0.5 * log(variance) - 0.5 * ((x - mean)^2 / variance)
            # (We drop the -0.5 * log(2*pi) constant as it's the same for all classes)
            
            term1 = math.log(var)
            term2 = ((x - mean) ** 2) / var
            
            log_prob -= 0.5 * (term1 + term2)
            
        # Keep track of the highest probability
        if log_prob > max_log_prob:
            max_log_prob = log_prob
            best_class = c
            
    return best_class

# --- Example Usage on Board ---
# features = [1.2, -0.5, 3.4, ...] # The 39 MFCC features extracted from the mic
# result_idx = predict_audio_class(features)
# print("Predicted Word:", model_data.CLASSES[result_idx])


# --- Example Usage on Board ---
# ['on', 'off', 'stop', 'uknown']
i = 16
# features = [-2.5686844e+02,  1.3959177e+02,  8.0665617e+00, -4.1448826e+01, -2.9944046e+01,  3.2988396e-01,  6.2178426e+00,  9.0643892e+00, -6.5838027e+00,  5.9275532e+00, -2.8192373e+01,  1.9626240e+01, -1.0130860e+00, -2.5470102e+02,  1.6171916e+02, -9.4717093e+00, -1.6660915e+01,  9.5316715e+00,  1.5823692e+01, -4.0831656e+00, -1.6109245e+01, -1.6403475e+01, -2.7065601e+00, -7.6615257e+00, 1.2409652e+01, -2.4190521e+01, -4.9443311e+02,  8.9877983e+01, 1.8117041e+01,  3.1036968e+01,  5.0818684e+01,  3.1483164e+01, 4.5603323e+00, -1.2117879e+01, -1.4944232e+01,  2.7241716e+00, 7.9998846e+00, -1.2753278e+01, -1.6924435e+01] # 13 features from mic
import random
for j in range(10):
    i = random.randint(0, len(y_test))
    features = X_test[i]
    result_idx = predict_audio_class(features)
    print("Predicted Word:", model_data.CLASSES[result_idx], "and actuall is:", y_test[i] + 1)

Predicted Word: on and actuall is: 1
Predicted Word: stop and actuall is: 3
Predicted Word: unknown and actuall is: 3
Predicted Word: stop and actuall is: 3
Predicted Word: unknown and actuall is: 1
Predicted Word: unknown and actuall is: 4
Predicted Word: off and actuall is: 2
Predicted Word: stop and actuall is: 2
Predicted Word: unknown and actuall is: 4
Predicted Word: off and actuall is: 2


*LDA + Logistic regression*

In [11]:
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import os

DATA_PATH = '../Data/'
OUTPUT_PATH = '../edge_mcu'

# 1. Load the FULL 39-feature dataset
X = np.load(os.path.join(DATA_PATH, 'X_features.npy'))
y = np.load(os.path.join(DATA_PATH, 'y_labels.npy'))

print(f"Loaded Data -> X shape: {X.shape}, y shape: {y.shape}")

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. STAGE 1: Dimensionality Reduction using LDA
print("\n--- Running LDA to reduce 39 features to 3 Super-Features ---")
# n_components is always (number of classes - 1) for LDA
lda = LinearDiscriminantAnalysis(n_components=3)
X_train_lda = lda.fit_transform(X_train, y_train)
X_test_lda = lda.transform(X_test)

print(f"New Training Data Shape after LDA: {X_train_lda.shape}")

# 3. STAGE 2: Optuna Optimization for Logistic Regression
print("\n--- Starting Optuna optimization for Logistic Regression ---")
def objective(trial):
    # Tune the regularization parameter C
    c_val = trial.suggest_float('C', 1e-4, 1e2, log=True)
    
    lr = LogisticRegression(C=c_val, max_iter=5000, random_state=42)
    lr.fit(X_train_lda, y_train)
    preds = lr.predict(X_test_lda)
    
    return accuracy_score(y_test, preds)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

print(f"\nBest Optuna Parameters: {study.best_params}")
print(f"Best Accuracy during CV: {study.best_value * 100:.2f}%")

# 4. TRAIN FINAL CLASSIFIER
print("\n--- Training Final Lightweight Classifier ---")
final_lr = LogisticRegression(**study.best_params, max_iter=5000, random_state=42)
final_lr.fit(X_train_lda, y_train)

preds = final_lr.predict(X_test_lda)
unique_classes = np.unique(y_test)
target_names = ['on', 'off', 'stop', 'unknown'][:len(unique_classes)]

print("\nFinal Classification Report:")
print(classification_report(y_test, preds, target_names=target_names))

# 5. EXPORT PURE PYTHON ARRAYS FOR EDGE INFERENCE
print("\n--- Exporting LDA and Classifier Parameters to model_data_lr.py ---")

# Extract LDA parameters
lda_xbar = lda.xbar_.tolist()         # Shape: (39,) -> The mean of each feature
lda_scalings = lda.scalings_.tolist() # Shape: (39, 3) -> The transformation matrix

# Extract Logistic Regression parameters
lr_coef = final_lr.coef_.tolist()           # Shape: (4, 3) -> Weights for the 3 super-features
lr_intercept = final_lr.intercept_.tolist() # Shape: (4,) -> Bias for each class

output_file = os.path.join(OUTPUT_PATH, 'model_data_lr.py')
with open(output_file, 'w') as f:
    f.write("# Auto-Generated LDA + Logistic Regression Export\n\n")
    f.write(f"CLASSES = {target_names}\n\n")
    
    # Write LDA arrays
    f.write("LDA_XBAR = [\n")
    f.write(f"    {[round(float(val), 6) for val in lda_xbar]}\n")
    f.write("]\n\n")
    
    f.write("LDA_SCALINGS = [\n")
    for row in lda_scalings:
        f.write(f"    {[round(float(val), 6) for val in row]},\n")
    f.write("]\n\n")
    
    # Write LR arrays
    f.write("LR_COEF = [\n")
    for class_weights in lr_coef:
        f.write(f"    {[round(float(w), 6) for w in class_weights]},\n")
    f.write("]\n\n")
    
    f.write("LR_INTERCEPT = [\n")
    f.write(f"    {[round(float(i), 6) for i in lr_intercept]}\n")
    f.write("]\n")

print(f"Pipeline successfully exported to: {output_file}")

[I 2026-07-14 12:14:53,180] A new study created in memory with name: no-name-6109fdda-e38c-4163-ba20-5805e4db1d38


[I 2026-07-14 12:14:53,194] Trial 0 finished with value: 0.762301918265221 and parameters: {'C': 0.06083748601580064}. Best is trial 0 with value: 0.762301918265221.
[I 2026-07-14 12:14:53,202] Trial 1 finished with value: 0.7597998331943286 and parameters: {'C': 0.0002953570999774}. Best is trial 0 with value: 0.762301918265221.
[I 2026-07-14 12:14:53,215] Trial 2 finished with value: 0.7614678899082569 and parameters: {'C': 0.019054305932245406}. Best is trial 0 with value: 0.762301918265221.
[I 2026-07-14 12:14:53,228] Trial 3 finished with value: 0.762301918265221 and parameters: {'C': 0.07113859797472444}. Best is trial 0 with value: 0.762301918265221.
[I 2026-07-14 12:14:53,239] Trial 4 finished with value: 0.7614678899082569 and parameters: {'C': 12.605702784282702}. Best is trial 0 with value: 0.762301918265221.
[I 2026-07-14 12:14:53,247] Trial 5 finished with value: 0.7589658048373644 and parameters: {'C': 0.0006120156303024916}. Best is trial 0 with value: 0.762301918265221.

Loaded Data -> X shape: (5995, 24), y shape: (5995,)

--- Running LDA to reduce 39 features to 3 Super-Features ---
New Training Data Shape after LDA: (4796, 3)

--- Starting Optuna optimization for Logistic Regression ---


[I 2026-07-14 12:14:53,368] Trial 14 finished with value: 0.7614678899082569 and parameters: {'C': 0.22915375867768514}. Best is trial 0 with value: 0.762301918265221.
[I 2026-07-14 12:14:53,380] Trial 15 finished with value: 0.7572977481234362 and parameters: {'C': 0.002954039297243885}. Best is trial 0 with value: 0.762301918265221.
[I 2026-07-14 12:14:53,396] Trial 16 finished with value: 0.7614678899082569 and parameters: {'C': 1.641800442826175}. Best is trial 0 with value: 0.762301918265221.
[I 2026-07-14 12:14:53,411] Trial 17 finished with value: 0.762301918265221 and parameters: {'C': 0.06831842691736253}. Best is trial 0 with value: 0.762301918265221.
[I 2026-07-14 12:14:53,423] Trial 18 finished with value: 0.7572977481234362 and parameters: {'C': 0.0025111712666229242}. Best is trial 0 with value: 0.762301918265221.
[I 2026-07-14 12:14:53,437] Trial 19 finished with value: 0.7614678899082569 and parameters: {'C': 2.189865243180451}. Best is trial 0 with value: 0.76230191826


Best Optuna Parameters: {'C': 0.00013102510391447037}
Best Accuracy during CV: 76.40%

--- Training Final Lightweight Classifier ---

Final Classification Report:
              precision    recall  f1-score   support

          on       0.75      0.77      0.76       299
         off       0.77      0.82      0.79       300
        stop       0.86      0.78      0.82       300
     unknown       0.69      0.69      0.69       300

    accuracy                           0.76      1199
   macro avg       0.77      0.76      0.76      1199
weighted avg       0.77      0.76      0.76      1199


--- Exporting LDA and Classifier Parameters to model_data_lr.py ---
Pipeline successfully exported to: ../edge_mcu\model_data_lr.py


In [12]:
import sys
import os

sys.path.append(os.path.abspath('../edge_mcu'))

import model_data_lr as model_data

def predict_audio_class(mfcc_features):
    """
    Two-stage inference:
    1. LDA Transform: Reduces 39 features to 3.
    2. Logistic Regression: Predicts the class based on the 3 features.
    """
    num_original_features = len(model_data.LDA_XBAR[0])
    num_super_features = len(model_data.LDA_SCALINGS[0])
    num_classes = len(model_data.CLASSES)
    
    # --- STAGE 1: LDA Transformation ---
    # Step 1A: Center the data (subtract the mean)
    centered_features = [0.0] * num_original_features
    for i in range(num_original_features):
        centered_features[i] = mfcc_features[i] - model_data.LDA_XBAR[0][i]
        
    # Step 1B: Matrix multiplication (Centered Data * Scalings)
    lda_features = [0.0] * num_super_features
    for i in range(num_super_features):
        sum_val = 0.0
        for j in range(num_original_features):
            sum_val += centered_features[j] * model_data.LDA_SCALINGS[j][i]
        lda_features[i] = sum_val
        
    # --- STAGE 2: Logistic Regression ---
    class_scores = [0.0] * num_classes
    
    for c in range(num_classes):
        # Start with the bias/intercept for this class
        score = model_data.LR_INTERCEPT[0][c]
        
        # Dot product: Super-features * Class Weights
        for i in range(num_super_features):
            score += lda_features[i] * model_data.LR_COEF[c][i]
            
        class_scores[c] = score
        
    # Find the index of the highest score (Argmax)
    best_class = 0
    max_score = class_scores[0]
    for i in range(1, num_classes):
        if class_scores[i] > max_score:
            max_score = class_scores[i]
            best_class = i
            
    return best_class

# --- Example Usage on Board ---
# features = [1.2, -0.5, 3.4, ...] # The 39 MFCC features extracted from the mic
# result_idx = predict_audio_class(features)
# print("Predicted Word:", model_data.CLASSES[result_idx])

import random
for j in range(10):
    i = random.randint(0, len(y_test))
    features = X_test[i]
    result_idx = predict_audio_class(features)
    print("Predicted Word:", model_data.CLASSES[result_idx], "and actuall is:", y_test[i] + 1)

Predicted Word: on and actuall is: 4
Predicted Word: on and actuall is: 1
Predicted Word: stop and actuall is: 3
Predicted Word: stop and actuall is: 3
Predicted Word: off and actuall is: 1
Predicted Word: off and actuall is: 2
Predicted Word: unknown and actuall is: 3
Predicted Word: unknown and actuall is: 1
Predicted Word: on and actuall is: 1
Predicted Word: on and actuall is: 1


LDA + Logistic regression
Targeted training

In [13]:
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import os

DATA_PATH = '../Data/'
OUTPUT_PATH = '../edge_mcu/'

# Ensure the Output directory exists
os.makedirs(OUTPUT_PATH, exist_ok=True)

# 1. Load the FULL 39-feature dataset
X = np.load(os.path.join(DATA_PATH, 'X_features.npy'))
y = np.load(os.path.join(DATA_PATH, 'y_labels.npy'))

print(f"Loaded Data -> X shape: {X.shape}, y shape: {y.shape}")

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. STAGE 1 & 2: Optuna Optimization for LDA & Logistic Regression combined
print("\n--- Starting Optuna optimization with Focus on Targets ---")
def objective(trial):
    # --- LDA PARAMETERS ---
    lda_shrinkage = trial.suggest_float('lda_shrinkage', 0.0, 1.0)
    lda = LinearDiscriminantAnalysis(solver='eigen', shrinkage=lda_shrinkage, n_components=3)
    
    X_train_lda = lda.fit_transform(X_train, y_train)
    X_test_lda = lda.transform(X_test)
    
    # --- LOGISTIC REGRESSION PARAMETERS ---
    c_val = trial.suggest_float('C', 1e-4, 1e2, log=True)
    
    # --- CLASS PRIORITIZATION ---
    target_weight = trial.suggest_float('target_weight', 1.0, 5.0)
    class_weights = {0: target_weight, 1: target_weight, 2: target_weight, 3: 1.0}
    
    lr = LogisticRegression(C=c_val, class_weight=class_weights, max_iter=5000, random_state=42)
    lr.fit(X_train_lda, y_train)
    preds = lr.predict(X_test_lda)
    
    return accuracy_score(y_test, preds)

# Run optimization
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

print(f"\nBest Optuna Parameters: {study.best_params}")
print(f"Best Accuracy during CV: {study.best_value * 100:.2f}%")

# 3. TRAIN FINAL PIPELINE WITH BEST PARAMETERS
print("\n--- Training Final Lightweight & Focused Classifier ---")

best_shrinkage = study.best_params['lda_shrinkage']
best_C = study.best_params['C']
best_target_weight = study.best_params['target_weight']

# Train Final LDA (using 'eigen')
final_lda = LinearDiscriminantAnalysis(solver='eigen', shrinkage=best_shrinkage, n_components=3)
X_train_lda_final = final_lda.fit_transform(X_train, y_train)
X_test_lda_final = final_lda.transform(X_test)

# Train Final Logistic Regression
final_weights = {0: best_target_weight, 1: best_target_weight, 2: best_target_weight, 3: 1.0}
final_lr = LogisticRegression(C=best_C, class_weight=final_weights, max_iter=5000, random_state=42)
final_lr.fit(X_train_lda_final, y_train)

# Evaluate
preds = final_lr.predict(X_test_lda_final)
unique_classes = np.unique(y_test)
target_names = ['on', 'off', 'stop', 'unknown'][:len(unique_classes)]

print("\nFinal Classification Report:")
print(classification_report(y_test, preds, target_names=target_names))

# 4. EXPORT PURE PYTHON ARRAYS FOR EDGE INFERENCE
print("\n--- Exporting LDA and Classifier Parameters ---")

# Extract LDA parameters (No xbar_ needed for 'eigen' solver!)
lda_scalings = final_lda.scalings_.tolist() 

# Extract Logistic Regression parameters
lr_coef = final_lr.coef_.tolist()           
lr_intercept = final_lr.intercept_.tolist() 

output_file = os.path.join(OUTPUT_PATH, 'model_data_lr.py')
with open(output_file, 'w') as f:
    f.write("# Auto-Generated Robust LDA (Eigen) + Logistic Regression Export\n\n")
    f.write(f"CLASSES = {target_names}\n\n")
    
    f.write("LDA_SCALINGS = [\n")
    for row in lda_scalings:
        f.write(f"    {[round(float(val), 6) for val in row]},\n")
    f.write("]\n\n")
    
    f.write("LR_COEF = [\n")
    for class_weights in lr_coef:
        f.write(f"    {[round(float(w), 6) for w in class_weights]},\n")
    f.write("]\n\n")
    
    f.write("LR_INTERCEPT = [\n")
    f.write(f"    {[round(float(i), 6) for i in lr_intercept]}\n")
    f.write("]\n")

print(f"Pipeline successfully exported to: {output_file}")

[I 2026-07-14 12:14:58,426] A new study created in memory with name: no-name-c207ff51-9eb2-4845-a92f-c3b0108ba896
[I 2026-07-14 12:14:58,466] Trial 0 finished with value: 0.7030859049207673 and parameters: {'lda_shrinkage': 0.5189111712377877, 'C': 0.2544956473145837, 'target_weight': 2.1919339280155543}. Best is trial 0 with value: 0.7030859049207673.
[I 2026-07-14 12:14:58,492] Trial 1 finished with value: 0.6914095079232694 and parameters: {'lda_shrinkage': 0.06876989960298863, 'C': 0.002456985453963993, 'target_weight': 3.991032940541674}. Best is trial 0 with value: 0.7030859049207673.
[I 2026-07-14 12:14:58,537] Trial 2 finished with value: 0.7197664720600501 and parameters: {'lda_shrinkage': 0.2913682857121207, 'C': 0.07752636176629034, 'target_weight': 2.1442210333391802}. Best is trial 2 with value: 0.7197664720600501.
[I 2026-07-14 12:14:58,569] Trial 3 finished with value: 0.7064220183486238 and parameters: {'lda_shrinkage': 0.15323477270144048, 'C': 0.16650784164220672, 'ta

Loaded Data -> X shape: (5995, 24), y shape: (5995,)

--- Starting Optuna optimization with Focus on Targets ---


[I 2026-07-14 12:14:58,667] Trial 6 finished with value: 0.5863219349457881 and parameters: {'lda_shrinkage': 0.7183521489404614, 'C': 0.0006311096537191445, 'target_weight': 4.84759664876208}. Best is trial 2 with value: 0.7197664720600501.
[I 2026-07-14 12:14:58,706] Trial 7 finished with value: 0.6330275229357798 and parameters: {'lda_shrinkage': 0.11546054046264731, 'C': 0.00014168625195583605, 'target_weight': 4.284243193150639}. Best is trial 2 with value: 0.7197664720600501.
[I 2026-07-14 12:14:58,751] Trial 8 finished with value: 0.6355296080066722 and parameters: {'lda_shrinkage': 0.6393955601079387, 'C': 0.005108043154275522, 'target_weight': 4.832639502905689}. Best is trial 2 with value: 0.7197664720600501.
[I 2026-07-14 12:14:58,783] Trial 9 finished with value: 0.6455379482902419 and parameters: {'lda_shrinkage': 0.0690553275474558, 'C': 0.00039268638014261606, 'target_weight': 3.613889079375365}. Best is trial 2 with value: 0.7197664720600501.
[I 2026-07-14 12:14:58,843]


Best Optuna Parameters: {'lda_shrinkage': 0.036330386975774474, 'C': 8.588555274910316, 'target_weight': 1.2968311325906416}
Best Accuracy during CV: 76.40%

--- Training Final Lightweight & Focused Classifier ---

Final Classification Report:
              precision    recall  f1-score   support

          on       0.75      0.78      0.76       299
         off       0.78      0.81      0.80       300
        stop       0.82      0.79      0.81       300
     unknown       0.70      0.67      0.68       300

    accuracy                           0.76      1199
   macro avg       0.76      0.76      0.76      1199
weighted avg       0.76      0.76      0.76      1199


--- Exporting LDA and Classifier Parameters ---
Pipeline successfully exported to: ../edge_mcu/model_data_lr.py


In [15]:
import model_data_lr

def predict_audio_class(mfcc_features):
    """
    Super-optimized Inference Engine for MicroPython (LDA Eigen + Logistic Regression).
    File size: < 3KB | Execution time: < 2ms
    """
    num_original_features = 24 # 13 MFCCs * 3 Time Frames
    num_super_features = 3     # Reduced dimensions by LDA
    num_classes = len(model_data_lr.CLASSES)
    
    # --- STAGE 1: LDA Transformation (Direct Matrix Multiplication) ---
    # Since we used 'eigen' solver, we directly multiply raw features by SCALINGS
    lda_features = [0.0] * num_super_features
    for i in range(num_super_features):
        sum_val = 0.0
        for j in range(num_original_features):
            sum_val += mfcc_features[j] * model_data_lr.LDA_SCALINGS[j][i]
        lda_features[i] = sum_val
        
    # --- STAGE 2: Logistic Regression Inference ---
    class_scores = [0.0] * num_classes
    for c in range(num_classes):
        # Start with the bias/intercept for this class
        score = model_data_lr.LR_INTERCEPT[0][c]
        
        # Dot product: Super-features * Class Weights
        for i in range(num_super_features):
            score += lda_features[i] * model_data_lr.LR_COEF[c][i]
            
        class_scores[c] = score
        
    # Find the index of the highest score (Argmax)
    best_class = 0
    max_score = class_scores[0]
    for i in range(1, num_classes):
        if class_scores[i] > max_score:
            max_score = class_scores[i]
            best_class = i
            
    return best_class

# --- Example Usage on Board ---
# features = [1.2, -0.5, 3.4, ...] # 39 features extracted from mic (VAD applied)
# result_idx = predict_audio_class(features)
# print("Predicted Word:", model_data_lr.CLASSES[result_idx])

import random
for j in range(10):
    i = random.randint(0, len(y_test))
    features = X_test[i]
    result_idx = predict_audio_class(features)
    print("Predicted Word:", model_data.CLASSES[result_idx], "and actuall is:", y_test[i] + 1)

Predicted Word: on and actuall is: 1
Predicted Word: on and actuall is: 4
Predicted Word: stop and actuall is: 3
Predicted Word: on and actuall is: 1
Predicted Word: stop and actuall is: 3
Predicted Word: stop and actuall is: 3
Predicted Word: on and actuall is: 1
Predicted Word: on and actuall is: 2
Predicted Word: on and actuall is: 2
Predicted Word: unknown and actuall is: 4
